# Visualisasi Artefak PCA — PDAM Pump Sentinel

Notebook ini mendemonstrasikan proses pelatihan **PCA T²/Q** dan visualisasi artefaknya.
Seluruh contoh menggunakan dataset surrogate **SKAB** (water circulation testbed) dalam skala minimal;
**bukan data operasional PDAM nyata**. Tujuannya adalah untuk pembelajaran akademik dan verifikasi pipeline.


## Dataset Demo

Kita menggunakan `tests/fixtures/skab_tiny.csv` — subset kecil dari SKAB dengan 3 baris telemetry.
File ini memastikan notebook dapat dijalankan secara mandiri tanpa akses ke dataset lengkap.


In [ ]:
import sys
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        fixture = candidate / 'tests' / 'fixtures' / 'skab_tiny.csv'
        if (candidate / 'pyproject.toml').exists() and fixture.exists():
            return candidate
    raise FileNotFoundError('Project root with tests/fixtures/skab_tiny.csv was not found')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INPUT_CSV = PROJECT_ROOT / 'tests' / 'fixtures' / 'skab_tiny.csv'
ARTIFACT_DIR = Path('/tmp/pdam-pump-sentinel-pca-demo/artifacts')

print('Project root:', PROJECT_ROOT)
print('Input CSV :', INPUT_CSV.resolve())
print('Artifact dir:', ARTIFACT_DIR.resolve())


## Pelatihan PCA T²/Q

Konfigurasi pelatihan menggunakan `window_size=1` dan `stride=1` agar cocok dengan dataset mini.
Threshold ditentukan pada kuantil 0.95; MLflow dinonaktifkan agar tidak bergantung pada server tracking.


In [ ]:
from ml.training.train_pca import PcaTrainingConfig, train_pca_from_skab

train_config = PcaTrainingConfig(
    input_path=INPUT_CSV,
    output_dir=ARTIFACT_DIR,
    window_size=1,
    stride=1,
    threshold_quantile=0.95,
    log_mlflow=False,
)

train_result = train_pca_from_skab(train_config)
print('Output dir     :', train_result.output_dir)
print('Artifact paths :', list(train_result.artifact_paths.keys()))
print('Thresholds     :', train_result.thresholds)


## Generasi Visualisasi

Membuat artefak visualisasi (Plotly JSON & HTML) dari hasil pelatihan di atas.


In [ ]:
from ml.visualization.pca_artifacts import (
    PcaArtifactVisualizationConfig,
    generate_pca_artifact_visualizations,
)

viz_config = PcaArtifactVisualizationConfig(
    artifact_dir=ARTIFACT_DIR,
)

viz_artifacts = generate_pca_artifact_visualizations(viz_config)
print('Visualisasi yang dihasilkan:')
for name, path in viz_artifacts.items():
    print(f"  {name}: {path}")


## Ringkasan Hasil

Menampilkan ringkasan metrik dan threshold dalam bentuk tabel pandas.


In [ ]:
import json

import pandas as pd

summary = json.loads(viz_artifacts['summary'].read_text())

df_summary = pd.DataFrame([
    {
        'sample_count': summary['metrics']['sample_count'],
        'anomaly_count': summary['metrics']['anomaly_count'],
        'normal_count': summary['metrics']['normal_count'],
        't2_threshold': summary['thresholds']['t2_threshold'],
        'q_threshold': summary['thresholds']['q_threshold'],
        'artifact_count': summary['artifact_count'],
    }
])

df_summary


## Grafik Scatter T²/Q

Membaca artefak Plotly JSON dan menampilkannya secara offline melalui `IPython.display.HTML`.
Preview SVG statis berikut dirender dari fixture mini agar grafik langsung terlihat saat notebook dibuka.

<svg width='720' height='360' viewBox='0 0 720 360' xmlns='http://www.w3.org/2000/svg' role='img' aria-label='Scatter PCA T2 Q'>
  <rect width='720' height='360' fill='#ffffff'/>
  <text x='24' y='34' font-size='20' font-family='Arial' font-weight='700'>Scatter T²/Q — fixture SKAB mini</text>
  <line x1='80' y1='300' x2='660' y2='300' stroke='#333333' stroke-width='1.5'/>
  <line x1='80' y1='60' x2='80' y2='300' stroke='#333333' stroke-width='1.5'/>
  <text x='330' y='338' font-size='13' font-family='Arial'>T² statistic</text>
  <text x='22' y='220' font-size='13' font-family='Arial' transform='rotate(-90 22 220)'>Q / SPE statistic</text>
  <line x1='660' y1='60' x2='660' y2='300' stroke='#3498db' stroke-width='2' stroke-dasharray='7 5'/>
  <line x1='80' y1='282' x2='660' y2='282' stroke='#9b59b6' stroke-width='2' stroke-dasharray='7 5'/>
  <text x='545' y='54' font-size='12' font-family='Arial' fill='#3498db'>T² threshold</text>
  <text x='88' y='276' font-size='12' font-family='Arial' fill='#9b59b6'>Q threshold</text>
  <circle cx='659' cy='294' r='8' fill='#2ecc71'><title>normal — 2024-01-01T00:00:00Z</title></circle>
  <circle cx='82' cy='76' r='8' fill='#e74c3c'><title>anomali label — 2024-01-01T00:00:01Z</title></circle>
  <circle cx='660' cy='292' r='8' fill='#2ecc71' stroke='#e74c3c' stroke-width='3'><title>normal diprediksi anomali — 2024-01-01T00:00:02Z</title></circle>
  <rect x='476' y='248' width='198' height='66' fill='#ffffff' stroke='#dddddd'/>
  <circle cx='494' cy='266' r='6' fill='#2ecc71'/><text x='508' y='270' font-size='12' font-family='Arial'>normal</text>
  <circle cx='494' cy='286' r='6' fill='#e74c3c'/><text x='508' y='290' font-size='12' font-family='Arial'>label anomali</text>
  <circle cx='494' cy='306' r='6' fill='#2ecc71' stroke='#e74c3c' stroke-width='2'/><text x='508' y='310' font-size='12' font-family='Arial'>diprediksi anomali</text>
</svg>


In [ ]:
import plotly.io as pio
from IPython.display import HTML

scatter_json = viz_artifacts['scores_scatter_plotly_json']
fig_scatter = pio.read_json(str(scatter_json))
HTML(fig_scatter.to_html(include_plotlyjs=True, full_html=False))


## Grafik Timeline

Menampilkan timeline T², Q, dan score beserta garis threshold.
Preview SVG statis memakai nilai ternormalisasi agar tiga seri dapat dibaca pada satu bidang.

<svg width='720' height='360' viewBox='0 0 720 360' xmlns='http://www.w3.org/2000/svg' role='img' aria-label='Timeline PCA score'>
  <rect width='720' height='360' fill='#ffffff'/>
  <text x='24' y='34' font-size='20' font-family='Arial' font-weight='700'>Timeline T²/Q/Score — fixture SKAB mini</text>
  <line x1='80' y1='300' x2='660' y2='300' stroke='#333333' stroke-width='1.5'/>
  <line x1='80' y1='60' x2='80' y2='300' stroke='#333333' stroke-width='1.5'/>
  <line x1='80' y1='90' x2='660' y2='90' stroke='#3498db' stroke-width='2' stroke-dasharray='7 5'/>
  <line x1='80' y1='282' x2='660' y2='282' stroke='#9b59b6' stroke-width='2' stroke-dasharray='7 5'/>
  <polyline points='100,92 380,286 660,90' fill='none' stroke='#3498db' stroke-width='3'/>
  <polyline points='100,296 380,78 660,294' fill='none' stroke='#9b59b6' stroke-width='3'/>
  <polyline points='100,92 380,286 660,90' fill='none' stroke='#f39c12' stroke-width='3' stroke-dasharray='4 4'/>
  <circle cx='100' cy='92' r='5' fill='#3498db'/><circle cx='380' cy='286' r='5' fill='#3498db'/><circle cx='660' cy='90' r='5' fill='#3498db'/>
  <circle cx='100' cy='296' r='5' fill='#9b59b6'/><circle cx='380' cy='78' r='5' fill='#9b59b6'/><circle cx='660' cy='294' r='5' fill='#9b59b6'/>
  <text x='74' y='324' font-size='11' font-family='Arial'>00:00:00</text>
  <text x='354' y='324' font-size='11' font-family='Arial'>00:00:01</text>
  <text x='634' y='324' font-size='11' font-family='Arial'>00:00:02</text>
  <rect x='486' y='70' width='178' height='76' fill='#ffffff' stroke='#dddddd'/>
  <line x1='500' y1='88' x2='536' y2='88' stroke='#3498db' stroke-width='3'/><text x='544' y='92' font-size='12' font-family='Arial'>T²</text>
  <line x1='500' y1='110' x2='536' y2='110' stroke='#9b59b6' stroke-width='3'/><text x='544' y='114' font-size='12' font-family='Arial'>Q / SPE</text>
  <line x1='500' y1='132' x2='536' y2='132' stroke='#f39c12' stroke-width='3' stroke-dasharray='4 4'/><text x='544' y='136' font-size='12' font-family='Arial'>score</text>
</svg>


In [ ]:
timeline_json = viz_artifacts['scores_timeline_plotly_json']
fig_timeline = pio.read_json(str(timeline_json))
HTML(fig_timeline.to_html(include_plotlyjs=True, full_html=False))


## Catatan

- Notebook ini dirancang untuk dijalankan **offline**; Plotly JS disertakan inline pada setiap output HTML.
- Dataset `skab_tiny.csv` hanya berisi 3 baris sehingga hasil visualisasi bersifat ilustratif,
  bukan representasi performa model pada data operasional sesungguhnya.
- Untuk eksperimen lebih lanjut, gunakan dataset SKAB lengkap atau data telemetry pompa PDAM yang valid.
